# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [ ]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.4/148.4 kB 9.1 MB/s eta 0:00:00


## 1) Define KB

In [ ]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [ ]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    #To-Do: you can customize the name and description of the tool here for example:
    # name = "kb_lookup_tool"
    # description = "Looks up relevant information from a custom knowledge base."

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {#To-Do: for inputs, you can customize the parameter names and types}
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()


## 3) Model (tiny local)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    #To-Do: set up the model parameters as needed
)

print("Model ready:", MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [10]:
!pip install -q smolagents[transformers]

from smolagents import Tool, TransformersModel, ToolCallingAgent

# From 43fdd3ba
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]

# From 7d266823
class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {"query": {"type": "string", "description": "The query string to look up in the knowledge base."}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {"a": {"type": "number", "description": "The first number."}, "b": {"type": "number", "description": "The second number."}, "op": {"type": "string", "optional": True, "nullable": True, "default": "add", "description": "The operation to perform (add or multiply)."}}
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

# From 2df59c54
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    #To-Do: set up the model parameters as needed
)

agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions=(
        "You are an agentic AI that uses tools to answer questions. Always cite your sources and keep answers concise (2-4 sentences)."
    ),
)

print(agent)

/usr/local/lib/python3.12/dist-packages/smolagents/models.py:933: FutureWarning: The 'model_id' parameter will be required in version 2.0.0. Please update your code to pass this parameter to avoid future errors. For now, it defaults to 'HuggingFaceTB/SmolLM2-1.7B-Instruct'.
  warnings.warn(


config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

## 5) Test queries

In [11]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - HuggingFaceTB/SmolLM2-1.7B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': 12, 'b': 30, 'op': 'add'}                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Error while parsing tool call from model output: The model output does not contain any JSON blob.

[Step 2: Duration 4.10 seconds| Input tokens: 2,519 | Output tokens: 263]

Reached max steps.

[Step 3: Duration 6.95 seconds| Input tokens: 2,989 | Output tokens: 469]

Answer: An agentic AI loop is a type of feedback loop in artificial intelligence where an AI system continuously interacts with its environment, receives feedback, and adjusts its behavior to achieve a desired outcome. This loop is characterized by the AI system acting as an agent, making decisions, and then receiving feedback from the environment, which it uses to adjust its actions and continue the loop.

In an agentic AI loop, the AI system is not simply reacting to external stimuli, but rather is actively engaged in a dynamic process of exploration, learning, and adaptation. The loop is self-sustaining, as the AI system's actions and decisions influence the feedback it receives, which in turn informs its future actions.

Agentic AI loops are commonly used in various applications, such as natural language processing, computer vision, and robotics, where the AI system needs to interact with humans or other agents in a dynamic environment. The loop allows the AI system to learn from i

### Re-running Test Queries

In [12]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)

---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - HuggingFaceTB/SmolLM2-1.7B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': 12, 'b': 30, 'op': 'add'}                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 42

...

Error while parsing tool call from model output: The model output does not contain any JSON blob.

[Step 2: Duration 4.02 seconds| Input tokens: 2,519 | Output tokens: 263]

Reached max steps.

[Step 3: Duration 6.86 seconds| Input tokens: 2,989 | Output tokens: 469]

Answer: An agentic AI loop is a type of feedback loop in artificial intelligence where an AI system continuously interacts with its environment, receives feedback, and adjusts its behavior to achieve a desired outcome. This loop is characterized by the AI system acting as an agent, making decisions, and then receiving feedback from the environment, which it uses to adjust its actions and continue the loop.

In an agentic AI loop, the AI system is not simply reacting to external stimuli, but rather is actively engaged in a dynamic process of exploration, learning, and adaptation. The loop is self-sustaining, as the AI system's actions and decisions influence the feedback it receives, which in turn informs its future actions.

Agentic AI loops are commonly used in various applications, such as natural language processing, computer vision, and robotics, where the AI system needs to interact with humans or other agents in a dynamic environment. The loop allows the AI system to learn from i